# Kaggle v6 overlap40 正式训练

用于 `dataset_v6_random811_overlap40` 主实验。默认运行 LTL-Net，ResNet50 encoder，seed=42，80 epochs。

两个 Kaggle 账号都可以使用本 notebook；数据路径会从候选列表中自动选择第一个存在的 overlap40 数据集。


In [ ]:
import os
import sys
import subprocess
import importlib.util
import importlib.metadata
import json
from pathlib import Path

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'
REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')

DATA_CANDIDATES = [
    Path('/kaggle/input/datasets/yuanssy/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datasets/changyasong/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datasets/changyasong/v6data/dataset_v6_random811_overlap40'),
]

print('Python:', sys.version)
print('Kaggle input roots:')
for p in Path('/kaggle/input').glob('*'):
    print(' -', p)


## 环境检查


In [ ]:
required = [
    ('rasterio', 'rasterio'),
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if importlib.util.find_spec('segmentation_models_pytorch') is None or smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import torch
import rasterio
import segmentation_models_pytorch as smp

assert torch.cuda.is_available(), 'Kaggle 当前没有开启 GPU，请在 Notebook settings 里选择 GPU。'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('SMP:', smp.__version__)
print('Rasterio:', rasterio.__version__)
x = torch.randn(1024, 1024, device='cuda')
print('CUDA 冒烟测试:', (x @ x).mean().item())
del x
torch.cuda.empty_cache()


## 获取代码


In [ ]:
if not REPO_DIR.exists():
    subprocess.check_call([
        'git', 'clone', '--branch', REPO_BRANCH, '--single-branch',
        REPO_URL, str(REPO_DIR)
    ])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'目录存在但不是 Git 仓库: {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])

assert (PROJECT_DIR / 'scripts' / 'train_ltl.py').is_file(), PROJECT_DIR
commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
branch = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--abbrev-ref', 'HEAD'], text=True
).strip()
print('项目目录:', PROJECT_DIR)
print('Git branch:', branch)
print('Git commit:', commit)


## 数据版本核验


In [ ]:
import numpy as np

EXPECTED_TILES = {'train': 1598, 'val': 200, 'test': 200}
EXPECTED_MEAN = np.array([
    0.15665339073973303, 0.6052870962271574, 0.22171011101838023,
    0.5087022443378417, 0.46687463729626205,
])
EXPECTED_STD = np.array([
    0.07239327406001447, 0.35159567816693277, 0.23999408652260576,
    0.18305312443820845, 0.18653673179588806,
])

def tif_files(folder):
    return sorted([*folder.glob('*.tif'), *folder.glob('*.tiff')])

existing = [p for p in DATA_CANDIDATES if p.is_dir()]
assert existing, ('没有找到 v6 overlap40 数据集，请检查 Kaggle Add data 是否挂载正确。候选路径: ' + '; '.join(map(str, DATA_CANDIDATES)))
DATA_ROOT = existing[0]
print('使用数据集:', DATA_ROOT)

for split, expected in EXPECTED_TILES.items():
    images = tif_files(DATA_ROOT / split / 'image')
    masks = tif_files(DATA_ROOT / split / 'mask')
    assert len(images) == expected, f'{split} image 数={len(images)}, 预期={expected}'
    assert len(masks) == expected, f'{split} mask 数={len(masks)}, 预期={expected}'
    assert {p.stem for p in images} == {p.stem for p in masks}, f'{split} image-mask 文件名不匹配'
    mask_by_stem = {p.stem: p for p in masks}
    for index in sorted({0, len(images) // 2, len(images) - 1}):
        with rasterio.open(images[index]) as src:
            image = src.read()
        with rasterio.open(mask_by_stem[images[index].stem]) as src:
            mask = src.read(1)
        assert image.shape == (5, 512, 512), (images[index], image.shape)
        assert mask.shape == (512, 512), (mask_by_stem[images[index].stem], mask.shape)
        bad = ~np.isfinite(image) | (image < -1e10)
        invalid_ratio = float(np.any(bad, axis=0).mean())
        if invalid_ratio > 0:
            print(f'{split} 提示: {images[index].name} 含 NoData/NaN 像元 {invalid_ratio:.4%}，训练时会按 MyDataset 逻辑置零并转为背景')
        assert invalid_ratio <= 0.05, f'无效像元比例过高: {images[index]}, ratio={invalid_ratio:.4%}'
        assert set(np.unique(mask)).issubset({0, 1, 2, 3, 4}), np.unique(mask)
    print(f'{split}: {len(images)} 对，抽检通过')

stats_path = DATA_ROOT / 'normalization_stats.json'
assert stats_path.is_file(), stats_path
stats = json.loads(stats_path.read_text(encoding='utf-8'))
assert np.allclose(stats['mean'], EXPECTED_MEAN, rtol=0, atol=1e-12), stats['mean']
assert np.allclose(stats['std'], EXPECTED_STD, rtol=0, atol=1e-12), stats['std']
print('数据版本核验通过')
print(json.dumps(stats, ensure_ascii=False, indent=2))


## 训练配置


In [ ]:
MODEL_KIND = 'ltl'          # 'ltl' 或 'baseline'
BASELINE_MODEL = 'DeepLabV3Plus'
ENCODER = 'resnet50'
SEED = 42
EPOCHS = 80
MAX_STEPS = 0              # 正式训练为 0；冒烟测试可设为 3 或 5
NUM_WORKERS = 2
RUN_SUFFIX = 'formal80'

assert MODEL_KIND in {'ltl', 'baseline'}
model_name = 'LTLNet' if MODEL_KIND == 'ltl' else BASELINE_MODEL
run_name = f'v6_overlap40_{model_name}_seed{SEED}_{RUN_SUFFIX}'
result_dir = OUTPUT_ROOT / f'result_{run_name}'
assert not result_dir.exists(), (
    f'结果目录已存在，为防止覆盖停止: {result_dir}; '
    '如需重跑，请修改 RUN_SUFFIX。'
)

common = [
    '--encoder', ENCODER,
    '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--seed', str(SEED),
    '--epochs', str(EPOCHS),
    '--max-steps', str(MAX_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--run-name', run_name,
]
if MODEL_KIND == 'ltl':
    command = [sys.executable, str(PROJECT_DIR / 'scripts' / 'train_ltl.py'), *common]
else:
    command = [
        sys.executable, str(PROJECT_DIR / 'scripts' / 'train_baseline.py'),
        '--model', BASELINE_MODEL, *common
    ]

print('运行命令:')
print(' '.join(command))
print('结果目录:', result_dir)


## 开始训练


In [ ]:
subprocess.check_call(command, cwd=PROJECT_DIR)


## 结果检查


In [ ]:
metrics_path = result_dir / 'metrics.json'
checkpoint_path = result_dir / 'best_model.pth'
assert metrics_path.is_file(), metrics_path
assert checkpoint_path.is_file(), checkpoint_path
result = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(result, ensure_ascii=False, indent=2))
print('结果目录:', result_dir)
print('best model:', checkpoint_path)


## 统一测试集评估、可视化与打包

重新加载最佳权重，在固定 Test 上生成混淆矩阵、逐类指标和代表/最好/困难案例。代表样本名单可供后续模型复用。


In [ ]:
evaluation_dir = result_dir / 'evaluation'
eval_command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'evaluate_segmentation.py'),
    '--model', model_name,
    '--encoder', ENCODER,
    '--data-dir', str(DATA_ROOT),
    '--checkpoint', str(checkpoint_path),
    '--output-dir', str(evaluation_dir),
    '--num-workers', str(NUM_WORKERS),
    '--samples-per-group', '4',
]
print('运行统一评估:')
print(' '.join(eval_command))
subprocess.check_call(eval_command, cwd=PROJECT_DIR)


In [ ]:
import shutil
archive_base = Path('/kaggle/working') / result_dir.name
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=result_dir)
print('完整结果压缩包:', archive_path)
print('请下载该 ZIP；其中包含权重、指标、训练历史、曲线、混淆矩阵和定性结果。')
